# C2 - SWELL-KW CONFOUND CONTROLS (step 8)

SWELL-KW only. Step 7 gave strong results (AUC up to 0.86). Before believing them we test three suspicions:

1. **Session order:** the neutral condition (N) is always Block 1, so "N vs pressure" is mixed up with "early vs late in the session".
2. **`dev_` features:** they are empty in the first 10 minutes (all N), and median-filling can reveal "early in session".
3. **`SnAppChange`:** interruptions work by switching applications, so this feature may simply restate the experimental manipulation.

## Controls
- **E1 (order-only models):** can we predict the label from `Blok` or minutes-since-start alone? If yes, the N-vs-pressure task is confounded.
- **E2 (T vs I only):** only Blocks 2 and 3, where T and I are counterbalanced across people (some do T first, some I first). Order cannot explain this one. A `Blok`-only model is the control for it.
- **E3 (dev_ without imputation):** drop rows where the baseline does not exist yet.
- **E4 (without app-switching features):** remove `SnAppChange_mean`, `SnTabfocusChange_mean`, `dev_SnAppChange_mean`.

Rule fixed in advance: "clearly above chance" if `mean AUC > shuffled mean + 2 x shuffled std`. People are never split between train and test. Preliminary work. Not validation of C2.

## Cell 1 - Load the step 7 table

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import re
import numpy as np
import pandas as pd
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

OUT_DIR = Path('/content/drive/MyDrive/AnchorAI_c2_outputs')
tbl = pd.read_csv(OUT_DIR / 'swell_model_table.csv', parse_dates=['time'])
tbl = tbl.sort_values(['PP', 'time']).reset_index(drop=True)
tbl['minutes_since_start'] = (tbl['time'] - tbl.groupby('PP')['time'].transform('min')).dt.total_seconds() / 60
META = ['PP', 'Blok', 'Condition', 'time', 'y_pressure', 'minutes_since_start']
FEATURES = [c for c in tbl.columns if c not in META]
print('rows:', len(tbl), '| people:', tbl['PP'].nunique(), '| features:', len(FEATURES))
print('\nWhich condition happens in which block (rows):')
print(pd.crosstab(tbl['Condition'], tbl['Blok']).to_string())
print('\nOrder of T and I across people (block 2 -> block 3):')
order = tbl[tbl['Blok'].isin([2, 3])].groupby(['PP', 'Blok'])['Condition'].first().unstack()
print(order.value_counts().to_string())

## Cell 2 - Helper: grouped cross-validation with a shuffled-label control
5 different fold layouts x 5 folds = 25 test folds. Preparation steps are inside the pipeline.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_validate

def make_model(name):
    if name == 'LR':
        return Pipeline([('imp', SimpleImputer(strategy='median')), ('scale', StandardScaler()),
                         ('clf', LogisticRegression(max_iter=2000, random_state=42))])
    return Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('clf', RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=42, n_jobs=-1))])

SEEDS = [0, 1, 2, 3, 4]
def cv_auc(df, cols, label, model_name, shuffled=False):
    aucs = []
    for s in SEEDS:
        y = df[label]
        if shuffled:
            y = pd.Series(np.random.RandomState(100 + s).permutation(y.values), index=y.index)
        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=s)
        r = cross_validate(make_model(model_name), df[cols], y, groups=df['PP'], cv=cv, scoring='roc_auc')
        aucs += list(r['test_score'])
    return np.array(aucs)

ROWS = []
def evaluate(exp, desc, df, cols, label):
    for m in ['LR', 'RF']:
        real = cv_auc(df, cols, label, m)
        shuf = cv_auc(df, cols, label, m, shuffled=True)
        thr = shuf.mean() + 2 * shuf.std()
        ROWS.append({'experiment': exp, 'what': desc, 'model': m, 'n_rows': len(df), 'n_features': len(cols),
                     'AUC mean': real.mean(), 'AUC std': real.std(), 'shuffled mean': shuf.mean(),
                     'clearly above chance?': 'YES' if real.mean() > thr else 'no'})
print('helper ready')

## Cell 3 - E1: can ORDER ALONE predict "N vs pressure"?
If `Blok` or `minutes_since_start` alone gives AUC near 1, then the N-vs-pressure task cannot be separated from session order.

In [ ]:
evaluate('E1', 'N vs T/I using Blok only', tbl, ['Blok'], 'y_pressure')
evaluate('E1', 'N vs T/I using minutes_since_start only', tbl, ['minutes_since_start'], 'y_pressure')
e1 = pd.DataFrame([r for r in ROWS if r['experiment'] == 'E1'])
print(e1.round(3).to_string(index=False))

## Cell 4 - E2: T vs I only (Blocks 2 and 3, order is counterbalanced)
Label `y_I = 1` for interruptions (I), `0` for time pressure (T). Order cannot explain this, so this is the cleanest test.
Feature sets: all, computer only, computer **without app-switching**, physiology only, `dev_` only.
Control: `Blok`-only model (should be near 0.5 if the design is balanced).

In [ ]:
ti = tbl[tbl['Condition'].isin(['T', 'I'])].copy()
ti['y_I'] = (ti['Condition'] == 'I').astype(int)
APP = ['SnAppChange_mean', 'SnTabfocusChange_mean', 'dev_SnAppChange_mean']
D_FEATS = [f for f in FEATURES if re.match(r'(dev_)?(HR|RMSSD|SCL)', f) or f == 'hr_valid_frac']
A_ONLY = [f for f in FEATURES if f not in D_FEATS]
sets_ti = {
    'Blok only (order control)': ['Blok'],
    'A+D all': FEATURES,
    'A only (computer)': A_ONLY,
    'A only WITHOUT app-switching': [f for f in A_ONLY if f not in APP],
    'D only (physiology)': D_FEATS,
    'dev_ only': [f for f in FEATURES if f.startswith('dev_')],
}
print('T vs I rows:', len(ti), '| class balance (1 = I):', ti['y_I'].value_counts().to_dict(), '| people:', ti['PP'].nunique())
for name, cols in sets_ti.items():
    evaluate('E2', f'T vs I: {name}', ti, cols, 'y_I')
e2 = pd.DataFrame([r for r in ROWS if r['experiment'] == 'E2'])
print(e2.round(3).drop(columns=['experiment']).to_string(index=False))

## Cell 5 - E3 and E4: N vs pressure again, with the suspicious parts removed
- E3: drop rows where the personal baseline (`dev_`) does not exist yet (no imputation of them).
- E4: also remove the app-switching features.
**Remember:** N is always Block 1, so these are still order-confounded. Compare them with the E1 order-only AUC.

In [ ]:
DEV_NONPHYS = [f for f in FEATURES if f.startswith('dev_') and not re.match(r'dev_(HR|RMSSD)', f)]
n3 = tbl.dropna(subset=DEV_NONPHYS).copy()
print('E3 rows after dropping rows with no baseline yet:', len(n3), 'of', len(tbl), '| classes:', n3['y_pressure'].value_counts().to_dict())
evaluate('E3', 'N vs T/I, dev_ only (non-physio), no-baseline rows dropped', n3, DEV_NONPHYS, 'y_pressure')
evaluate('E3', 'N vs T/I, A+D all, no-baseline rows dropped', n3, FEATURES, 'y_pressure')
evaluate('E4', 'N vs T/I, all features WITHOUT app-switching', tbl, [f for f in FEATURES if f not in APP], 'y_pressure')

nt = tbl[tbl['Condition'].isin(['N', 'T'])].copy()
nt['y_T'] = (nt['Condition'] == 'T').astype(int)
evaluate('E4', 'N vs T only (no app-switching difference), all features without app-switching', nt, [f for f in FEATURES if f not in APP], 'y_T')
evaluate('E4', 'N vs T only, Blok only (order control)', nt, ['Blok'], 'y_T')
e34 = pd.DataFrame([r for r in ROWS if r['experiment'] in ('E3', 'E4')])
print(e34.round(3).drop(columns=['experiment']).to_string(index=False))

## Cell 6 - Summary table and picture

In [ ]:
res = pd.DataFrame(ROWS)
res.round(4).to_csv(OUT_DIR / 'swell_confound_controls.csv', index=False)
print(res[['experiment', 'what', 'model', 'AUC mean', 'AUC std', 'shuffled mean', 'clearly above chance?']].round(3).to_string(index=False))

import matplotlib.pyplot as plt
rf = res[res['model'] == 'RF'].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(rf))
ax.barh(x, rf['AUC mean'], xerr=rf['AUC std'], color='tab:blue', alpha=0.8)
ax.axvline(0.5, color='black', linestyle='--', linewidth=1)
ax.set_yticks(x); ax.set_yticklabels([f"{r['experiment']}: {r['what']}"[:75] for _, r in rf.iterrows()], fontsize=7)
ax.invert_yaxis(); ax.set_xlim(0.4, 1.0); ax.set_xlabel('ROC-AUC (Random Forest, grouped CV)')
plt.tight_layout(); plt.savefig(OUT_DIR / 'swell_confound_controls.png', dpi=120); plt.show()

## Cell 7 - How to read this
- **E1 near 1.0** = N-vs-pressure is dominated by session order. Step 7's N-vs-pressure numbers cannot be read as stress detection.
- **E2 (T vs I) clearly above its `Blok`-only control** = a signal that order cannot explain. If it disappears once app-switching is removed, it only restated the interruption mechanism.
- **E2 physiology only** is the most C2-relevant (HR/HRV/SCL do not directly encode the manipulation).
- Claude reads these outputs directly from this tab.